In [ ]:
# setup (pip install python-dotenv if the import fails)
import os, glob, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    def load_dotenv():
        env_file = Path(".env")
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    name, value = line.split("=", 1)
                    os.environ.setdefault(name.strip(), value.strip())

load_dotenv()
KEY = os.getenv("EIA_API_KEY")
assert KEY, "No EIA key found. Check the variable name in your .env"

DATA = Path("Data")
YEAR = 2025   # same year as the RTO features, so the two tables line up
ROUTE = "electricity/electric-power-operational-data"

In [ ]:
# EIA API helpers: paging, automatic retries, one-state-at-a-time downloads
RETRY_CODES = {429, 500, 502, 503, 504}

def get_with_retry(url, params, tries=6):
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code not in RETRY_CODES:
                r.raise_for_status()
                return r.json()["response"]
            err = f"HTTP {r.status_code}"
        except (requests.ConnectionError, requests.Timeout) as e:
            err = type(e).__name__
        if attempt == tries:
            raise RuntimeError(f"EIA kept failing ({err}) after {tries} tries. Wait a few minutes and rerun this cell.")
        wait = 10 * attempt
        print(f"   {err}, retrying in {wait}s (try {attempt + 1}/{tries})")
        time.sleep(wait)

def eia_get(route, params):
    url = f"https://api.eia.gov/v2/{route}/data/"
    rows, offset = [], 0
    while True:
        resp = get_with_retry(url, {**params, "api_key": KEY, "offset": offset, "length": 5000})
        rows.extend(resp["data"])
        offset += 5000
        if offset >= int(resp["total"]):
            break
        time.sleep(0.2)
    return pd.DataFrame(rows)

def eia_get_by_state(route, params, states, cache_dir):
    """One state at a time, each saved to its own CSV. Rerun after a failure and it resumes."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    for i, st in enumerate(states, 1):
        f = cache_dir / f"{st}.csv"
        if f.exists():
            parts.append(pd.read_csv(f))
            continue
        df = eia_get(route, {**params, "facets[location][]": st})
        if df.empty:
            df = pd.DataFrame(columns=["period", "location", "fueltypeid", "generation"])
        df.to_csv(f, index=False)
        parts.append(df)
        print(f"[{i}/{len(states)}] {st}: {len(df):,} rows")
    return pd.concat(parts, ignore_index=True)

In [ ]:
# load the raw atlas with the same cleaning as the generator/RTO joins
RAW = DATA / "im3_2025_atlas.csv"

ID_DTYPES = {"id": str, "state_id": str, "county_id": str}  # keep leading zeros
base = pd.read_csv(RAW, dtype=ID_DTYPES)

print(base.shape)             # expect about (1242, 14)
print(base.columns.tolist())  # must include state_abb, state_id, county_id, sqft, lat, lon

base = base.drop_duplicates()
base = base.drop_duplicates(subset=["id"]).reset_index(drop=True)
base = base[base["sqft"] > 0]
base = base[
    base["lat"].between(17.0, 72.0) & base["lon"].between(-170.0, -65.0)
].reset_index(drop=True)

N_BASE = len(base)
assert base["id"].is_unique
print(N_BASE, "facilities (should be 1147)")

(1242, 13)
['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type']
1147 facilities (should be 1147)


In [ ]:
# quick test: one state, to check the route, filters and column names
test = eia_get_by_state(ROUTE, {
    "frequency": "monthly",
    "data[]": "generation",
    "facets[sectorid][]": "99",
    "start": f"{YEAR}-01",
    "end": f"{YEAR}-12",
    "sort[0][column]": "period", "sort[0][direction]": "asc",
}, ["TX"], DATA / f"eia_state_gen_{YEAR}")

print(test.shape)
print(test.columns.tolist())
print(test["fueltypeid"].value_counts())
test.head()

[1/1] TX: 433 rows
(433, 9)
['period', 'location', 'stateDescription', 'sectorid', 'sectorDescription', 'fueltypeid', 'fuelTypeDescription', 'generation', 'generation-units']
fueltypeid
ALL    12
AOR    12
BIO    12
COL    12
COW    12
DFO    12
DPV    12
FOS    12
HYC    12
LFG    12
LIG    12
MLG    12
NG     12
NGO    12
NUC    12
OB2    12
OBW    12
OOG    12
ORW    12
OTH    12
PC     12
PEL    12
PET    12
RC     12
REN    12
RFO    12
SPV    12
SUB    12
SUN    12
TPV    12
TSN    12
WAS    12
WND    12
WNT    12
WOO    12
WWW    12
ANT     1
Name: count, dtype: int64


,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025-01,TX,Texas,99,All Sectors,ALL,all fuels,50550.79569,thousand megawatthours
1,2025-01,TX,Texas,99,All Sectors,ANT,anthracite coal,0,thousand megawatthours
2,2025-01,TX,Texas,99,All Sectors,AOR,all renewables,13852.38101,thousand megawatthours
3,2025-01,TX,Texas,99,All Sectors,BIO,biomass,84.92311,thousand megawatthours
4,2025-01,TX,Texas,99,All Sectors,COL,"coal, excluding waste coal",7261.02864,thousand megawatthours


In [ ]:
# all states in the IM3 base, one at a time (finished states load from disk on reruns)
states = sorted(base["state_abb"].dropna().unique())
print(len(states), "states:", states)

gen = eia_get_by_state(ROUTE, {
    "frequency": "monthly",
    "data[]": "generation",
    "facets[sectorid][]": "99",
    "start": f"{YEAR}-01",
    "end": f"{YEAR}-12",
    "sort[0][column]": "period", "sort[0][direction]": "asc",
}, states, DATA / f"eia_state_gen_{YEAR}")

assert len(gen) > 0, "No rows came back. Check the sectorid and the date range."
print(f"{len(gen):,} rows")
print("Columns:", gen.columns.tolist())
if "generation-units" in gen.columns:
    print("Units:", gen["generation-units"].dropna().unique())
print("Fuel codes:", sorted(gen["fueltypeid"].dropna().unique()))

42 states: ['AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'FL', 'GA', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'WA', 'WI', 'WY']
[1/42] AL: 405 rows
[2/42] AR: 422 rows
[3/42] AZ: 384 rows
   HTTP 504, retrying in 10s (try 2/6)
   HTTP 504, retrying in 20s (try 3/6)
   HTTP 504, retrying in 30s (try 4/6)
   HTTP 504, retrying in 40s (try 5/6)
   HTTP 504, retrying in 50s (try 6/6)
[4/42] CA: 471 rows
[5/42] CO: 396 rows
   HTTP 504, retrying in 10s (try 2/6)
[6/42] CT: 422 rows
   HTTP 504, retrying in 10s (try 2/6)
   HTTP 504, retrying in 20s (try 3/6)
[7/42] FL: 433 rows
[8/42] GA: 408 rows
[9/42] IA: 408 rows
[10/42] ID: 396 rows
   HTTP 504, retrying in 10s (try 2/6)
[11/42] IL: 444 rows
[12/42] IN: 408 rows
   ReadTimeout, retrying in 10s (try 2/6)
[13/42] KS: 348 rows
   HTTP 504, retrying in 10s (try 2/6)
[14/42] KY: 360 rows
[15/42] MA: 408 row

In [ ]:
# clean, then one row per state with fuel-mix shares
COAL, GAS, NUCLEAR = ["COL"], ["NG"], ["NUC"]
HYDRO, SOLAR, WIND = ["HYC"], ["SUN"], ["WND"]
FOSSIL = ["FOS"]                                # EIA's all-fossil total
RENEWABLE = SOLAR + WIND + HYDRO + ["GEO"]      # same definition as the generator/RTO joins (no biomass)
UNIT_TO_MWH = 1000                              # data is in thousand MWh

g = (gen.reindex(columns=["period", "location", "fueltypeid", "generation"])
        .dropna(subset=["location", "fueltypeid"]).copy())
g["generation"] = pd.to_numeric(g["generation"], errors="coerce")
g = g.drop_duplicates(subset=["period", "location", "fueltypeid"])
g = g[g["generation"] > 0]

print("Months per state:", g.groupby("location")["period"].nunique().value_counts().to_dict())

annual = g.pivot_table(index="location", columns="fueltypeid", values="generation",
                       aggfunc="sum", fill_value=0)
total = annual["ALL"]

def share(codes):
    missing = [c for c in codes if c not in annual.columns]
    if missing:
        print(f"   codes not in data (ignored): {missing}")
    return annual[[c for c in codes if c in annual.columns]].sum(axis=1) / total

state_feat = pd.DataFrame({
    "st_share_coal":      share(COAL),
    "st_share_gas":       share(GAS),
    "st_share_nuclear":   share(NUCLEAR),
    "st_share_hydro":     share(HYDRO),
    "st_share_solar":     share(SOLAR),
    "st_share_wind":      share(WIND),
    "st_share_fossil":    share(FOSSIL),
    "st_share_renewable": share(RENEWABLE),
    "st_share_clean":     share(RENEWABLE + NUCLEAR),
    "st_total_gen_mwh":   total * UNIT_TO_MWH,
}).reset_index().rename(columns={"location": "state_abb"})

assert (state_feat.filter(like="st_share").fillna(0) <= 1.01).all().all()
state_feat.sort_values("st_total_gen_mwh", ascending=False).head(10)

Months per state: {12: 42}


,state_abb,st_share_coal,st_share_gas,st_share_nuclear,st_share_hydro,st_share_solar,st_share_wind,st_share_fossil,st_share_renewable,st_share_clean,st_total_gen_mwh
36,TX,0.116579,0.487676,0.070452,0.002110,0.099225,0.219077,0.608753,0.320412,0.390864,5.909221e+08
6,FL,0.038030,0.729306,0.114217,0.000270,0.093072,0.000000,0.771884,0.093342,0.207558,2.678037e+08
32,PA,0.042653,0.582705,0.304213,0.010762,0.007629,0.014500,0.657452,0.032892,0.337105,2.466991e+08
3,CA,0.001138,0.356150,0.085283,0.135294,0.269927,0.078961,0.362964,0.536379,0.621662,2.058844e+08
10,IL,0.159740,0.168383,0.504994,0.000158,0.033577,0.129316,0.329241,0.163051,0.668045,1.934086e+08
29,OH,0.243225,0.547037,0.116277,0.002936,0.053625,0.020179,0.805069,0.076741,0.193017,1.474231e+08
7,GA,0.136559,0.388101,0.349325,0.020389,0.076835,0.000000,0.526808,0.097225,0.446550,1.466392e+08
0,AL,0.152505,0.456347,0.300431,0.058269,0.009379,0.000000,0.608987,0.067647,0.368078,1.420382e+08
22,NC,0.132180,0.400313,0.317836,0.034330,0.094154,0.006913,0.533911,0.135397,0.453233,1.354369e+08
28,NY,0.000000,0.471972,0.215502,0.196011,0.039752,0.052362,0.481697,0.288125,0.503627,1.318203e+08


In [ ]:
# left join on state_abb: each facility gets its state's row
st_features = base[["id", "state_abb"]].merge(state_feat, on="state_abb", how="left")

assert len(st_features) == N_BASE, "row count changed during the join"
assert st_features["id"].is_unique, "duplicate ids after the join"

print("Share of missing values per column:")
print(st_features.isna().mean().round(3).sort_values(ascending=False).head())

st_features = st_features.drop(columns="state_abb")
st_features.to_csv(DATA / "eia_state_gen_features.csv", index=False)
print("Saved", DATA / "eia_state_gen_features.csv", st_features.shape)

Share of missing values per column:
id                  0.0
state_abb           0.0
st_share_coal       0.0
st_share_gas        0.0
st_share_nuclear    0.0
dtype: float64
Saved Data/eia_state_gen_features.csv (1147, 11)


In [41]:
# final table: facilities + retail price + generation mix + energy estimates
retail = pd.read_csv(DATA / "eia_retail_sales.csv")
retail = retail[["period", "stateid", "price", "sales"]].rename(columns={
    "stateid": "state_abb",
    "price": "comm_elec_price_cents_kwh",
    "sales": "comm_elec_sales_mwh",
})
for c in ["comm_elec_price_cents_kwh", "comm_elec_sales_mwh"]:
    retail[c] = pd.to_numeric(retail[c], errors="coerce")

retail_period = retail["period"].max()
retail_latest = retail[retail["period"] == retail_period].drop(columns="period")
print("Retail snapshot:", retail_period)

final = (base[["id", "state_abb", "county", "type", "sqft", "lat", "lon"]]
         .merge(retail_latest, on="state_abb", how="left")
         .merge(state_feat, on="state_abb", how="left"))

WATTS_PER_SQFT = 100
final["est_mwh_per_hour"] = final["sqft"] * WATTS_PER_SQFT / 1_000_000
final["est_hourly_energy_cost_usd"] = final["est_mwh_per_hour"] * final["comm_elec_price_cents_kwh"] * 10

assert len(final) == N_BASE and final["id"].is_unique
print(final.shape)
print(final.isna().mean().round(3).sort_values(ascending=False).head(5))

final.to_csv(DATA / "im3_multimodal_atlas_fueltype.csv", index=False)

Retail snapshot: 2026-07
(1147, 21)
id                  0.0
st_share_nuclear    0.0
est_mwh_per_hour    0.0
st_total_gen_mwh    0.0
st_share_clean      0.0
dtype: float64
